<a href="https://colab.research.google.com/github/Deboxa/Cats_dogs_ML_GRS/blob/main/ML_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install tensorflowjs

INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of wheel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting un

In [ ]:
import os, cv2
from tqdm import tqdm

SOURCE_DIR = '/content/drive/MyDrive/cat_n_dog_dataset'
DEST_DIR = '/content/dataset_processed/train'
IMG_SIZE = 100

os.makedirs(DEST_DIR, exist_ok=True)
for cls in ['Cat', 'Dog']:
    os.makedirs(os.path.join(DEST_DIR, cls), exist_ok=True)

for cls in ['Cat', 'Dog']:
    src_cls_dir = os.path.join(SOURCE_DIR, cls)
    dst_cls_dir = os.path.join(DEST_DIR, cls)
    for fname in tqdm(os.listdir(src_cls_dir), desc=cls):
        src = os.path.join(src_cls_dir, fname)
        try:
            img = cv2.imread(src, cv2.IMREAD_UNCHANGED)
            if img is None: continue
            if len(img.shape) == 3:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            elif len(img.shape) != 2:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            dst = os.path.join(dst_cls_dir, fname.rsplit('.',1)[0] + '.jpg')
            cv2.imwrite(dst, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        except Exception as e:
            print(f"Error {src}: {e}")
print("Done.")

Dog:   1%|          | 118/12499 [00:27<40:26,  5.10it/s]

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflowjs as tfjs

tf.random.set_seed(42)
IMG_SIZE = 100
BATCH_SIZE = 32
EPOCHS = 20
TRAIN_DIR = '/content/dataset_processed/train'

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    color_mode='grayscale', label_mode='binary')
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    color_mode='grayscale', label_mode='binary')

normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras', monitor='val_accuracy', save_best_only=True, mode='max')

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=[early_stop, checkpoint])

model.save('cats_dogs_final_model.keras')
tfjs.save_keras_model(model, 'tfjs_model')

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {val_acc*100:.2f}%")

ModuleNotFoundError: No module named 'tensorflowjs'